# 第4回講義 宿題

---

# Lecture 4: Homework

## 課題

今Lessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) を多層パーセプトロンによって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

---

## Task

Based on what you learned in this lesson, classify the fashion version of MNIST (Fashion MNIST, 10 classes) using a multilayer perceptron.

For details about Fashion MNIST, refer to the link below.

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値 (Target)

Accuracy 85%


### ルール


- 訓練データは`x_train`， `t_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train`，`t_train`以外の学習データは使わないでください．**
- Pytorchを利用して構いません．
- ただし，**`torch.nn.Conv2d`のような高レベルのAPIは使用しないで下さい**．具体的には，`nn.Parameter`, `nn.Module`, `nn.Sequential`以外の`nn`系のAPIです．使用した場合エラーになります．
- torchvision等で既に実装されているモデルも使用しないで下さい．

---

### Rules

- Training data are provided as `x_train` and `t_train`, and test data are provided as `x_test`.
- Output the predicted labels as class labels (0–9), not as one-hot vectors.
- **Do not use any training data other than the provided `x_train` and `t_train` specified in the cells below.**
- You may use PyTorch.
- However, **do not use high-level APIs such as `torch.nn.Conv2d`.** Concretely, this means any `nn.*` APIs other than `nn.Parameter`, `nn.Module`, and `nn.Sequential`. If you use them, an error will occur.
- Do not use pre-implemented models (e.g., those in torchvision).

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第4回 ニューラルネットワークの最適化・正則化」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第4回 ニューラルネットワークの最適化・正則化 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

---

### Submission
- Submit two items:
    1. Save your predicted labels for the test data (`x_test`) as a CSV file, and submit it by selecting **“第4回 ニューラルネットワークの最適化・正則化”** in the Omnicampus homework tab.
    2. Save the corresponding Python code by choosing **File → Download → Download .py**, and submit it by selecting **“第4回 ニューラルネットワークの最適化・正則化 (code)”** in the Omnicampus homework tab. Do not upload the .py file itself; instead, copy and paste the entire code into the “submission content” field.

- Grading is based on (1). Item (2) is used to verify your implementation (top submissions may be asked to share their code). If you change your code, **please resubmit both (1) and (2)**.

### 評価方法
- 予測ラベルの`t_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

---

### Evaluation

- We evaluate your submission by accuracy against the ground-truth labels `t_test`.
- Submissions are graded immediately and the leaderboard is updated (the grading schedule will be announced separately).
- Your score at the deadline will be used as the final evaluation.

### ドライブのマウント

---

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set the working directory
work_dir = 'drive/MyDrive/Colab Notebooks/DLBasics2026_colab'

### データの読み込み（この部分は修正しないでください）

`__len__`は，Pythonの組み込み関数len()を呼んだときに，内部で呼ばれる特殊メソッドです．

`__getitem__`は，インデックスやキーで要素を取得するときに，内部で呼ばれる特殊メソッドです．

どちらも， Datasetクラスを自作する際によく登場します．

---

### Loading the data (Do not modify this section)

`__len__` is a special method that is called internally when you use Python's built-in function `len()`.

`__getitem__` is a special method that is called internally when you access an element by index or key.

Both commonly appear when you implement your own `Dataset` class.

```python
class MyList:
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]

mylist = MyList([10, 20, 30])
print(len(mylist))  # Calls __len__
# 3
print(mylist[1])  # Calls __getitem__
# 20
```


In [ ]:
# Disable Dynamo because it conflicts with the API-restriction code
import torch._dynamo
torch._dynamo.disable()

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import inspect

# Restrict available APIs
nn_except = ["Module", "Parameter", "Sequential"]
for m in inspect.getmembers(nn):
    if not m[0] in nn_except and m[0][0:2] != "__":
        delattr(nn, m[0])

seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training data
x_train = np.load(work_dir + '/Lecture04/data/x_train.npy')
t_train = np.load(work_dir + '/Lecture04/data/y_train.npy')

# Test data
x_test = np.load(work_dir + '/Lecture04/data/x_test.npy')

class train_dataset(torch.utils.data.Dataset):
    def __init__(self, x_train, t_train):
        self.x_train = x_train.reshape(-1, 784).astype('float32') / 255
        self.t_train = t_train

    def __len__(self):
        return self.x_train.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.x_train[idx], dtype=torch.float), torch.tensor(self.t_train[idx], dtype=torch.long)

class test_dataset(torch.utils.data.Dataset):
    def __init__(self, x_test):
        self.x_test = x_test.reshape(-1, 784).astype('float32') / 255

    def __len__(self):
        return self.x_test.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.x_test[idx], dtype=torch.float)

trainval_data = train_dataset(x_train, t_train)
test_data = test_dataset(x_test)

### 多層パーセプトロンの実装

---

### Implement a Multilayer Perceptron

In [ ]:
batch_size = 32

val_size = 10000
train_size = len(trainval_data) - val_size

train_data, val_data = torch.utils.data.random_split(trainval_data, [train_size, val_size])

dataloader_train = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)

dataloader_valid = torch.utils.data.DataLoader(
    val_data,
    batch_size=batch_size,
    shuffle=False
)

dataloader_test = torch.utils.data.DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
def relu(x):
    # WRITE ME


def softmax(x):
    # WRITE ME


class Dense(nn.Module):  # Inherit from nn.Module
    # WRITE ME


class MLP(nn.Module):  # Inherit from nn.Module
    # WRITE ME

in_dim = 784
hid_dim = 200
out_dim = 10
lr = 0.001
n_epochs = 10


mlp = # WRITE ME

optimizer = # WRITE ME

In [ ]:
for epoch in range(n_epochs):
    losses_train = []
    losses_valid = []
    train_num = 0
    train_true_num = 0
    valid_num = 0
    valid_true_num = 0

    mlp.train()  # Training mode (compute gradients)
    for x, t in dataloader_train:
        # WRITE ME

        losses_train.append(loss.tolist())

        acc = torch.where(t - pred.to("cpu") == 0, torch.ones_like(t), torch.zeros_like(t))
        train_num += acc.size()[0]
        train_true_num += acc.sum().item()

    mlp.eval()  # Eval mode (do not compute gradients)
    for x, t in dataloader_valid:
        # WRITE ME

        losses_valid.append(loss.tolist())

        acc = torch.where(t - pred.to("cpu") == 0, torch.ones_like(t), torch.zeros_like(t))
        valid_num += acc.size()[0]
        valid_true_num += acc.sum().item()

    print('EPOCH: {}, Train [Loss: {:.3f}, Accuracy: {:.3f}], Valid [Loss: {:.3f}, Accuracy: {:.3f}]'.format(
        epoch,
        np.mean(losses_train),
        train_true_num/train_num,
        np.mean(losses_valid),
        valid_true_num/valid_num
    ))

In [ ]:
mlp.eval()

t_pred = []
for x in dataloader_test:

    x = x.to(device)

    # Forward propagation
    y = mlp.forward(x)

    # Convert the model output to scalar class predictions
    pred = y.argmax(1).tolist()

    t_pred.extend(pred)

submission = pd.Series(t_pred, name='label')
submission.to_csv(work_dir + '/Lecture04/submission_pred_04.csv', header=True, index_label='id')